# Estabilidade

Uma rede pode ter a arquitetura certa, a perda certa e o otimizador certo, e ainda assim não treinar. O sinal que atravessa muitas camadas cresce ou encolhe a cada uma delas, e o gradiente que volta faz o mesmo. Quando ele chega quase nulo às primeiras camadas, elas param de aprender; quando chega enorme, um único passo destrói os pesos.

Este material usa uma mesma rede de oito camadas no MNIST para mostrar o problema e as três peças que o resolvem: a inicialização dos pesos, a normalização das ativações e o decaimento da taxa de aprendizado.

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

In [ ]:
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

## Dados

O MNIST entra normalizado, como nos materiais anteriores. Os valores 0.1307 e 0.3081 são a média e o desvio padrão dos pixels no conjunto de treino, e a transformação $x' = (x - \mu) / \sigma$ deixa a entrada com média zero e desvio padrão um. É a primeira escala a controlar: se a entrada já chega grande ou pequena demais, tudo o que vem depois herda o problema.

Para que cada treinamento leve poucos segundos, usa-se um subconjunto de 4.000 exemplos de treino e 1.000 de validação.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.1307,), std=(0.3081,)),
])

full_train_set = datasets.MNIST(root="data", train=True, download=True, transform=transform)
full_test_set = datasets.MNIST(root="data", train=False, download=True, transform=transform)

In [ ]:
train_set = Subset(full_train_set, range(4000))
validation_set = Subset(full_test_set, range(1000))

train_dataloader = DataLoader(train_set, batch_size=64, shuffle=True)
validation_dataloader = DataLoader(validation_set, batch_size=500, shuffle=False)

images, labels = next(iter(train_dataloader))
images, labels = images.to(device), labels.to(device)
print(f"treino: {len(train_set)}, validação: {len(validation_set)}, lote: {tuple(images.shape)}")

## A rede

A rede é uma MLP com oito camadas ocultas de 64 unidades, mais profunda do que o MNIST precisa, de propósito: os problemas deste material aparecem com a profundidade. A função recebe a ativação como argumento e, opcionalmente, insere uma camada de normalização depois de cada `nn.Linear`. Essa opção fica desligada até a seção correspondente.

In [ ]:
def deep_mlp(activation, batch_norm=False, depth=8, width=64):
    layers = [nn.Flatten()]   # [batch, 1, 28, 28] -> [batch, 784]
    in_features = 28 * 28

    for _ in range(depth):
        layers.append(nn.Linear(in_features, width))
        if batch_norm:
            layers.append(nn.BatchNorm1d(width))
        layers.append(activation())
        in_features = width

    layers.append(nn.Linear(width, 10))   # [batch, 10]
    return nn.Sequential(*layers).to(device)

Duas funções acompanham a rede até o fim. `evaluate` mede a acurácia na validação, e `train` roda o laço de treinamento com SGD por dez épocas e devolve a acurácia de validação de cada época. O agendador é opcional e só entra na última seção.

In [ ]:
def evaluate(model, dataloader):
    model.eval()
    correct = 0

    with torch.no_grad():
        for batch_images, batch_labels in dataloader:
            batch_images, batch_labels = batch_images.to(device), batch_labels.to(device)
            correct += (model(batch_images).argmax(dim=1) == batch_labels).sum().item()

    return correct / len(dataloader.dataset)

In [ ]:
def train(model, lr, scheduler_class=None, epochs=10, **scheduler_kwargs):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    scheduler = scheduler_class(optimizer, **scheduler_kwargs) if scheduler_class else None
    accuracies = []

    for epoch in range(epochs):
        model.train()
        for batch_images, batch_labels in train_dataloader:
            batch_images, batch_labels = batch_images.to(device), batch_labels.to(device)
            loss = criterion(model(batch_images), batch_labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        if scheduler is not None:
            scheduler.step()

        accuracies.append(evaluate(model, validation_dataloader))
        print(f"[{epoch + 1}/{epochs}] acurácia de validação: {accuracies[-1]:.4f}")

    return accuracies

Cada treinamento acrescenta a sua curva ao dicionário `curves`, e `plot_accuracies` desenha todas as que já existem, para que cada versão da rede seja comparada com as anteriores.

In [ ]:
curves = {}


def plot_accuracies(curves):
    plt.figure(figsize=(8, 5))
    for name, accuracies in curves.items():
        plt.plot(range(1, len(accuracies) + 1), accuracies, marker="o", label=name)

    plt.xlabel("época")
    plt.ylabel("acurácia de validação")
    plt.ylim(0, 1)
    plt.legend()
    plt.grid(True)
    plt.show()

## Gradientes que somem e que explodem

Ao atravessar uma camada, o gradiente é multiplicado pela derivada da ativação e pelos pesos. Em uma rede com $n$ camadas, o gradiente que chega à primeira passou por $n$ multiplicações desse tipo, e o produto muda exponencialmente com a profundidade. Se os fatores são menores que um, ele encolhe até desaparecer: é o **gradiente que some**. Se são maiores que um, ele cresce sem controle: é o **gradiente que explode**.

A derivada da sigmoid vale no máximo $0.25$, e perto de zero nas pontas, então ela puxa o produto para baixo em toda camada. A da ReLU vale $1$ em todo o lado positivo, e não encolhe o sinal por conta própria. Os pesos podem empurrar nos dois sentidos.

A função abaixo faz um único `backward` e devolve a norma do gradiente dos pesos de cada camada, da primeira à última. Ela é aplicada a três redes: uma com sigmoid, uma com ReLU e uma com ReLU cujos pesos foram multiplicados por dez.

In [ ]:
def gradient_norms(model):
    loss = nn.CrossEntropyLoss()(model(images), labels)

    model.zero_grad()
    loss.backward()

    return [layer.weight.grad.norm().item() for layer in model if isinstance(layer, nn.Linear)]

In [ ]:
torch.manual_seed(0)
sigmoid_norms = gradient_norms(deep_mlp(nn.Sigmoid))

torch.manual_seed(0)
relu_norms = gradient_norms(deep_mlp(nn.ReLU))

torch.manual_seed(0)
large_model = deep_mlp(nn.ReLU)
with torch.no_grad():
    for layer in large_model:
        if isinstance(layer, nn.Linear):
            layer.weight.mul_(10)
large_norms = gradient_norms(large_model)

plt.figure(figsize=(8, 5))
plt.plot(sigmoid_norms, marker="o", label="sigmoid")
plt.plot(relu_norms, marker="o", label="ReLU")
plt.plot(large_norms, marker="o", label="ReLU, pesos x10")
plt.yscale("log")
plt.xlabel("camada")
plt.ylabel("norma do gradiente")
plt.legend()
plt.grid(True)
plt.show()

Com sigmoid, a norma cai seis ordens de grandeza entre a última camada e a primeira, e as camadas iniciais praticamente não são atualizadas. Com os pesos dez vezes maiores, ela passa de $10^{5}$ em todas as camadas, e um único passo de SGD levaria os pesos para qualquer lugar. A ReLU com os pesos padrão fica no meio, mas ainda perde um fator de vinte, e é isso que o treinamento dela mostra.

In [ ]:
torch.manual_seed(0)
relu_model = deep_mlp(nn.ReLU)
curves["ReLU"] = train(relu_model, lr=0.1)
plot_accuracies(curves)

A rede não sai dos 10% de acurácia, que é o acaso entre dez classes. Trocar a sigmoid pela ReLU não bastou: com a escala de pesos que o `nn.Linear` usa por padrão, o sinal ainda encolhe demais em oito camadas.

## Inicialização dos pesos

Antes da primeira iteração os pesos precisam de algum valor. A escolha mais ingênua é zerar tudo, e vale ver o que acontece com o gradiente nesse caso. A função `apply_init` aplica um inicializador do `nn.init` aos pesos de todas as camadas lineares, e zera os vieses.

In [ ]:
def apply_init(model, initializer):
    for layer in model:
        if isinstance(layer, nn.Linear):
            initializer(layer.weight)
            nn.init.zeros_(layer.bias)

In [ ]:
zero_model = deep_mlp(nn.Sigmoid)
apply_init(zero_model, nn.init.zeros_)

print([f"{norm:.4f}" for norm in gradient_norms(zero_model)])
print(zero_model[-1].weight.grad[:3, :4])

Só a última camada recebe gradiente. O gradiente que volta para as anteriores é multiplicado pelos pesos, que são zero, e nada chega. E mesmo a última não aprende nada útil: todas as suas entradas são iguais, já que cada unidade oculta calcula $\sigma(0) = 0.5$, e por isso as colunas do gradiente são idênticas. Depois do passo, as unidades de cada camada continuam iguais entre si, e nenhuma quantidade de treinamento quebra essa **simetria**. Inicializar com qualquer outra constante tem o mesmo problema.

A saída é sortear os pesos, mas a escala do sorteio decide se o sinal atravessa a rede, some ou explode. A receita de **He** escolhe a variância dos pesos em função do número de entradas da camada, de modo que a variância da saída fique igual à da entrada,

$$
\mathrm{Var}[w] = \frac{2}{n_{in}}
$$

em que $n_{in}$ é o número de entradas da camada. O fator $2$ compensa o fato de a ReLU zerar metade das ativações. A receita de **Xavier**, $\mathrm{Var}[w] = 2 / (n_{in} + n_{out})$, é a equivalente para ativações simétricas como a tanh; com ReLU ela perde metade da variância a cada camada.

A verificação é direta: passar um lote pela rede e medir o desvio padrão da saída de cada camada, com uma normal de desvio fixo pequeno, uma de desvio fixo grande e a de He.

In [ ]:
def activation_stds(model):
    stds = []
    x = images

    with torch.no_grad():
        for layer in model:
            x = layer(x)
            if isinstance(layer, nn.Linear):
                stds.append(x.std().item())

    return stds

In [ ]:
initializers = {
    "normal, desvio 0.05": lambda w: nn.init.normal_(w, mean=0.0, std=0.05),
    "normal, desvio 0.5": lambda w: nn.init.normal_(w, mean=0.0, std=0.5),
    "He": lambda w: nn.init.kaiming_normal_(w, nonlinearity="relu"),
}

plt.figure(figsize=(8, 5))
for name, initializer in initializers.items():
    torch.manual_seed(0)
    model = deep_mlp(nn.ReLU)
    apply_init(model, initializer)
    plt.plot(activation_stds(model), marker="o", label=name)

plt.yscale("log")
plt.xlabel("camada")
plt.ylabel("desvio padrão das ativações")
plt.legend()
plt.grid(True)
plt.show()

Os desvios fixos ignoram o tamanho da camada e erram nas duas direções: com 0.05 o sinal chega ao fim em $10^{-5}$, com 0.5 chega em $10^{4}$. A de He mantém a escala do começo ao fim. A inicialização padrão do `nn.Linear` é mais conservadora que a He, o que é inofensivo em redes rasas mas não em oito camadas. Com a He, a mesma rede treina.

In [ ]:
torch.manual_seed(0)
he_model = deep_mlp(nn.ReLU)
apply_init(he_model, initializers["He"])
curves["ReLU + He"] = train(he_model, lr=0.1)
plot_accuracies(curves)

A rede que estava parada nos 10% passa dos 90%, e nada mudou além dos valores iniciais dos pesos.

## Batch normalization

A inicialização acerta a escala no instante zero, mas os pesos mudam a cada passo, e com eles a escala das ativações internas. A **batch normalization** recalibra essa escala em toda passada, normalizando cada ativação dentro do mini lote,

$$
\hat{x}_i = \frac{x_i - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}
\qquad
y_i = \gamma \hat{x}_i + \beta
$$

em que $\mu_B$ e $\sigma_B^2$ são a média e a variância do lote, $\epsilon$ evita divisão por zero, e $\gamma$ e $\beta$ são parâmetros treináveis que devolvem à rede a liberdade de escolher a escala e o deslocamento.

A camada entra na pilha depois de cada `nn.Linear` e antes da ativação, que é o que o argumento `batch_norm` da `deep_mlp` faz. O teste mais duro é a rede com sigmoid, a que perdia seis ordens de grandeza.

In [ ]:
torch.manual_seed(0)
batch_norm_norms = gradient_norms(deep_mlp(nn.Sigmoid, batch_norm=True))

plt.figure(figsize=(8, 5))
plt.plot(sigmoid_norms, marker="o", label="sigmoid")
plt.plot(batch_norm_norms, marker="o", label="sigmoid + batch norm")
plt.yscale("log")
plt.xlabel("camada")
plt.ylabel("norma do gradiente")
plt.legend()
plt.grid(True)
plt.show()

A norma fica praticamente constante ao longo das oito camadas. Como a entrada de cada sigmoid é reposta na região central, onde a derivada é maior, a saturação deixa de se acumular, e a rede treina.

In [ ]:
torch.manual_seed(0)
batch_norm_model = deep_mlp(nn.Sigmoid, batch_norm=True)
curves["sigmoid + batch norm"] = train(batch_norm_model, lr=0.1)
plot_accuracies(curves)

A camada tem comportamentos diferentes no treino e na avaliação. Durante o treino ela usa as estatísticas do lote atual e vai acumulando uma média móvel delas; na avaliação usa a média acumulada, para que a previsão de um exemplo não dependa dos outros exemplos do lote. É o `model.train()` e o `model.eval()` que alternam entre os dois modos, e a função `evaluate` já faz essa troca. Esquecer o `eval()` é o erro mais comum com essa camada: um único exemplo em modo de treino nem chega a passar, porque não há como calcular a variância de um elemento só.

## Decaimento da taxa de aprendizado

Um passo grande ajuda no começo, quando os parâmetros estão longe de qualquer mínimo, e atrapalha no fim, quando falta apenas ajustar. Um agendador, ou **scheduler**, reduz a taxa de aprendizado ao longo do treinamento. O `StepLR` a divide por um fator fixo a cada tantas épocas; o `CosineAnnealingLR` a leva suavemente de $\eta_0$ até zero,

$$
\eta_t = \frac{\eta_0}{2} \left(1 + \cos \frac{\pi t}{T}\right)
$$

em que $t$ é a época e $T$ é o total de épocas. O agendador acompanha o otimizador, e o seu `step` é chamado uma vez por época, depois do laço dos lotes, que é o que a função `train` faz quando recebe um.

A rede final junta ReLU, inicialização de He e batch normalization. Com o sinal estabilizado ela tolera um passo maior, e a taxa inicial sobe para 0.5. O primeiro treinamento mantém essa taxa constante; o segundo a reduz com o cosseno.

In [ ]:
torch.manual_seed(0)
constant_model = deep_mlp(nn.ReLU, batch_norm=True)
apply_init(constant_model, initializers["He"])
curves["ReLU + He + batch norm"] = train(constant_model, lr=0.5)

In [ ]:
torch.manual_seed(0)
cosine_model = deep_mlp(nn.ReLU, batch_norm=True)
apply_init(cosine_model, initializers["He"])
curves["ReLU + He + batch norm + cosseno"] = train(cosine_model, lr=0.5, scheduler_class=torch.optim.lr_scheduler.CosineAnnealingLR, T_max=10)
plot_accuracies(curves)

Com a taxa constante, a acurácia continua subindo e descendo nas últimas épocas, porque o passo segue grande quando a rede já está perto de um mínimo. Com o decaimento, as últimas épocas ficam mais regulares e a curva termina no seu ponto mais alto.

Cada peça atuou em um momento diferente: a inicialização acerta a escala do sinal antes do primeiro passo, a batch normalization a mantém enquanto os pesos mudam, e o agendador reduz o passo quando o que falta é só ajuste fino. Nenhuma delas muda o que a rede é capaz de representar; elas só garantem que o gradiente chegue a todas as camadas com um tamanho útil. O que ainda separa esses pouco mais de 90% de uma acurácia maior já não é fluxo de sinal, e sim a diferença entre decorar os 4.000 exemplos de treino e generalizar para os outros. Esse é o assunto do próximo material.

## Exercícios

### Exercício 1

Repita o experimento das normas dos gradientes com `depth=4` e com `depth=16`. Como a profundidade muda a distância entre a curva da sigmoid e a da ReLU? A rede com ReLU e inicialização padrão, que não treinou com oito camadas, treina com quatro?

In [ ]:
depths = [4, 8, 16]

### Exercício 2

Repita o experimento da inicialização com zeros usando uma constante diferente de zero, `nn.init.constant_(w, 0.01)`. Agora todas as camadas recebem gradiente. Imprima as primeiras linhas de `model[1].weight.grad` e explique por que a rede continua sem conseguir treinar.

In [ ]:
# apply_init(model, lambda w: nn.init.constant_(w, 0.01))

### Exercício 3

Troque o `CosineAnnealingLR` pelo `ReduceLROnPlateau`, que não segue fórmula fixa: ele observa uma métrica e multiplica a taxa por `factor` quando ela para de melhorar por `patience` épocas. Como a métrica aqui é a acurácia, ele precisa de `mode="max"`, e o `step` precisa receber a acurácia da época, o que exige uma pequena mudança em `train`. Em que época ele decidiu reduzir a taxa?

In [ ]:
# scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)